# GRPO Training — All 4 NYT Games (Single Model, Simultaneous Trained)

In [ ]:
import os, sys

REPO_URL = "https://github.com/jackiepiepkorn/cse291-nytgames.git"
REPO_DIR = "/kaggle/working/cse291-nytgames"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
%pip install torch transformers datasets accelerate trl>=0.15 bitsandbytes gymnasium pandas --quiet
!pip install peft --quiet

In [ ]:
import re
import json
import random
import torch
from pathlib import Path
from datasets import Dataset, concatenate_datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
import sys, os
from peft import LoraConfig, PeftModel
from trl import GRPOConfig, GRPOTrainer

# %cd ..

from nytgames import (
    SpellingBeeConfig, SpellingBeeEnv, SpellingBeeDataset,
    WordleDatset, WordleConfig, WordleEnv, load_dictionary,
    StrandsConfig, StrandsEnv, StrandsDataset, 
    ConnectionsConfig, ConnectionsEnv, ConnectionsDataset
)
from nytgames.env.wordle import _score_guess, _format_feedback
from nytgames.data.dataset import _SPELLING_BEE_CSV_PATH as _SB_CSV_PATH, _WORDLE_PAST_SOLUTIONS_TXT_PATH, _STRANDS_CSV_PATH, _CONNECTIONS_CSV_PATH, _PROMPTS_DIR

# Prompts are class attributes on SpellingBeeDataset, not module-level vars
SB_SYSTEM_PROMPT = SpellingBeeDataset._SYSTEM_PROMPT
SB_USER_PROMPT_TEMPLATE = SpellingBeeDataset._USER_PROMPT_TEMPLATE

WORDLE_SYSTEM_PROMPT = WordleDataset._SYSTEM_PROMPT
WORDLE_USER_PROMPT_TEMPLATE = WordleDataset._USER_PROMPT_TEMPLATE

STRANDS_SYSTEM_PROMPT = StrandsDataset._SYSTEM_PROMPT
STRANDS_USER_PROMPT_TEMPLATE = StrandsDataset._USER_PROMPT_TEMPLATE

CONNECTIONS_SYSTEM_PROMPT = ConnectionsDataset._SYSTEM_PROMPT
CONNECTIONS_USER_PROMPT_TEMPLATE = ConnectionsDataset._USER_PROMPT_TEMPLATE

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [2]:
# GAME = "spelling_bee"
# model now trains on ALL games - no switch needed 

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./grpo_output"

NUM_GENERATIONS = 4
MAX_COMPLETION_LENGTH = 16   # spelling bee words are 4-15 letters; saves generation time
TEMPERATURE = 1.0            # slightly lower than 1.2 for better quality exploration
BETA = 0.04

LEARNING_RATE = 5e-6
NUM_EPOCHS = 1               # multiple passes over fewer puzzles for stronger signal
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 1
USE_BF16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False

USE_LORA = True
LORA_R = 16
LORA_ALPHA = 32

MAX_PUZZLES = 50           # more puzzles for better GRPO diversity (was 200)
# CSV_PATH = SB_CSV_PATH

# print(f"Game: {GAME}")
print(f"Model: {MODEL_NAME}")
print(f"BF16: {USE_BF16}, LoRA: {USE_LORA} (r={LORA_R})")
print(f"Group size G={NUM_GENERATIONS}, β={BETA}, lr={LEARNING_RATE}")
print(f"Puzzles: {MAX_PUZZLES or 'all'}, Epochs: {NUM_EPOCHS}, Max tokens: {MAX_COMPLETION_LENGTH}")

Model: Qwen/Qwen2.5-1.5B-Instruct
BF16: False, LoRA: True (r=16)
Group size G=4, β=0.04, lr=5e-06
Puzzles: 50, Epochs: 1, Max tokens: 16


## 3. Reward Functions

### 3.1 Spelling Bee Reward Functions

In [ ]:
def _extract_text(completion) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        for msg in reversed(completion):
            if isinstance(msg, dict) and msg.get("content"):
                return msg["content"]
        return ""
    return str(completion)

def spelling_bee_format_reward(completions, **kwargs) -> list[float]:
    game_types = kwargs.get("game_type", [])
    rewards = []
    for i, completion in enumerate(completions):
        if game_types and game_types[i]!= "spelling_bee":
            rewards.append(0.0)
            continue
        clean = _extract_text(completion).strip()
        if re.fullmatch(r"[A-Za-z]+", clean):
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


def spelling_bee_letter_reward(completions, **kwargs) -> list[float]:
    """1.0 ONLY if the word uses exclusively allowed letters, contains the center,
    and is at least 4 characters.  0.0 otherwise — no partial credit."""
    game_types = kwargs.get("game_type", [])

    letters_list = kwargs.get("letters", [])
    centers = kwargs.get("center", [])
    rewards = []
    for i, completion in enumerate(completions):
        if game_types and game_types[i]!= "spelling_bee":
            rewards.append(0.0)
            continue
        word = _extract_text(completion).strip().upper()
        allowed = set(letters_list[i].upper()) if isinstance(letters_list[i], str) else set(l.upper() for l in letters_list[i])
        center = centers[i].upper()

        if not re.fullmatch(r"[A-Za-z]+", word):
            rewards.append(0.0)
            continue

        uses_only_allowed = set(word) <= allowed
        has_center = center in word
        long_enough = len(word) >= 4

        # ALL THREE criteria must be met — no partial credit for wrong letters
        if uses_only_allowed and has_center and long_enough:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


def spelling_bee_dictionary_reward(completions, **kwargs) -> list[float]:
    game_types = kwargs.get("game_type", [])

    solutions_list = kwargs.get("valid_words", [])
    rewards = []
    for i, completion in enumerate(completions):
        if game_types and game_types[i]!= "spelling_bee":
            rewards.append(0.0)
            continue
        word = _extract_text(completion).strip().lower()
        solutions = set(s.strip().lower() for s in solutions_list[i].replace(";", ",").split(","))
        if word in solutions:
            rewards.append(3.0)  # strong signal for valid puzzle words
        else:
            rewards.append(0.0)
    return rewards


def spelling_bee_length_reward(completions, **kwargs) -> list[float]:
    game_types = kwargs.get("game_type", [])

    solutions_list = kwargs.get("valid_words", [])
    rewards = []
    for i, completion in enumerate(completions):
        if game_types and game_types[i]!= "spelling_bee":
            rewards.append(0.0)
            continue
        word = _extract_text(completion).strip().lower()
        solutions = set(s.strip().lower() for s in solutions_list[i].replace(";", ",").split(","))
        if word not in solutions:
            rewards.append(0.0)
            continue
        if len(word) == 4:
            score = 1.0
        else:
            score = float(len(word))
        rewards.append(score / 10.0)
    return rewards


def spelling_bee_pangram_reward(completions, **kwargs) -> list[float]:
    game_types = kwargs.get("game_type", [])

    letters_list = kwargs.get("letters", [])
    solutions_list = kwargs.get("valid_words", [])
    rewards = []
    for i, completion in enumerate(completions):
        if game_types and game_types[i]!= "spelling_bee":
            rewards.append(0.0)
            continue
        word = _extract_text(completion).strip().lower()
        solutions = set(s.strip().lower() for s in solutions_list[i].replace(";", ",").split(","))
        if word not in solutions:
            rewards.append(0.0)
            continue
        allowed = set(letters_list[i].lower()) if isinstance(letters_list[i], str) else set(l.lower() for l in letters_list[i])
        if set(word) >= allowed:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


### 3.2 Wordle Reward Functions

In [ ]:
def wordle_format_reward(completions, **kwargs) -> list[float]:
    game_types = kwargs.get("game_type", [])

    rewards = []
    for i, completion in enumerate(completions):
        if game_types and game_types[i] != "wordle":
            rewards.append(0.0)
            continue

        word = _extract_text(completion).strip()
        if re.fullmatch(r"[A-Za-z]{5}", word):
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


def wordle_letter_match_reward(completions, **kwargs) -> list[float]:
    game_types = kwargs.get("game_type", [])

    targets = kwargs.get("target_word", [])
    rewards = []
    for i, completion in enumerate(completions):
        if game_types and game_types[i] != "wordle":
            rewards.append(0.0)
            continue

        guess = _extract_text(completion).strip().upper()
        target = targets[i].upper()
        if len(guess) != 5:
            rewards.append(0.0)
            continue
        green = sum(1 for g, t in zip(guess, target) if g == t)
        remaining_target = [t for g, t in zip(guess, target) if g != t]
        yellow = 0
        for g, t in zip(guess, target):
            if g != t and g in remaining_target:
                yellow += 1
                remaining_target.remove(g)
        score = (green * 2.0 + yellow * 1.0) / 10.0
        rewards.append(score)
    return rewards


def wordle_exact_match_reward(completions, **kwargs) -> list[float]:
    game_types = kwargs.get("game_type", [])

    targets = kwargs.get("target_word", [])
    rewards = []
    for i, completion in enumerate(completions):
        if game_types and game_types[i] != "wordle":
            rewards.append(0.0)
            continue
        guess = _extract_text(completion).strip().upper()
        target = targets[i].upper()
        rewards.append(1.0 if guess == target else 0.0)
    return rewards

def wordle_constraint_reward(completions, **kwargs):
    """Reward the model for respecting what it already knows."""
    game_types = kwargs.get("game_type", [])
    known_greens_list = kwargs.get("known_greens", [])
    known_grays_list = kwargs.get("known_grays", [])
    rewards = []
    
    for i, completion in enumerate(completions):
        if game_types and game_types[i] != "wordle":
            rewards.append(0.0)
            continue
        guess = _extract_text(completion).strip().upper()
        if len(guess) != 5:
            rewards.append(0.0)
            continue
        
        score = 0.0
        try:
            greens = eval(known_greens_list[i]) if known_greens_list else {}
            grays = set(eval(known_grays_list[i])) if known_grays_list else set()
        except:
            greens, grays = {}, set()
        
        # +0.3 per green position respected
        for pos, letter in greens.items():
            if guess[pos] == letter:
                score += 0.3
        # -0.2 per gray letter reused
        for letter in grays:
            if letter in guess:
                score -= 0.2
        
        rewards.append(max(0.0, score))
    return rewards


### 3.3 Strands and Connections Reward Functions

In [4]:
# other reward functions
def strands_theme_reward(completions, **kwargs) -> list[float]:
    game_types = kwargs.get("game_type", [])
    rewards = []
    for i, completion in enumerate(completions):
        if game_types and game_types[i] != "strands":
            rewards.append(0.0)
            continue
        word = _extract_text(completion).strip().upper()
        theme_words = set(w.upper() for w in kwargs.get("theme_words", [""])[i].split("|") if w)
        spangram = kwargs.get("spangram", [""])[i].upper()
        if word == spangram:
            rewards.append(2.0)
        elif word in theme_words:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards

def connections_category_reward(completions, **kwargs) -> list[float]:
    game_types = kwargs.get("game_type", [])
    rewards = []
    for i, completion in enumerate(completions):
        if game_types and game_types[i] != "connections":
            rewards.append(0.0)
            continue
        text = _extract_text(completion).strip()
        guessed = frozenset(w.strip().upper() for w in text.split(","))
        categories_str = kwargs.get("categories_str", [""])[i]
        # Parse "CAT1:W1,W2,W3,W4|CAT2:..." back into groups
        correct = False
        for cat_entry in categories_str.split("|"):
            if ":" not in cat_entry:
                continue
            _, words_str = cat_entry.split(":", 1)
            cat_words = frozenset(w.strip().upper() for w in words_str.split(","))
            if guessed == cat_words:
                correct = True
                break
        rewards.append(1.0 if correct else 0.0)
    return rewards

### 3.4 Shared Reward Function

In [ ]:
# ---------- Shared reward: is it a real English word? ----------
VALID_ENGLISH_WORDS = load_dictionary()

def real_word_reward(completions, **kwargs) -> list[float]:
    """Reward 0.5 if the completion is any real English word (≥4 letters)."""
    rewards = []
    for completion in completions:
        word = _extract_text(completion).strip().upper()
        if len(word) >= 4 and word in VALID_ENGLISH_WORDS:
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards

In [6]:
# ONE reward_funcs list for ONE model
reward_funcs = [
    spelling_bee_format_reward,
    spelling_bee_letter_reward,
    spelling_bee_dictionary_reward,
    spelling_bee_length_reward,
    spelling_bee_pangram_reward,
    wordle_format_reward,
    wordle_letter_match_reward,
    wordle_exact_match_reward,
    strands_theme_reward,
    connections_category_reward,
    real_word_reward,
    wordle_constraint_reward
]

### 3.5 Test Reward Functions

In [19]:
test_completions = [
    [{"role": "assistant", "content": "throw"}],
    [{"role": "assistant", "content": "ARROW"}],
    [{"role": "assistant", "content": "xyz!!"}],
    [{"role": "assistant", "content": "thaw"}],
    [{"role": "assistant", "content": "thataway"}],
]
test_kwargs = {
    "letters": ["WAHORTY"] * 5,
    "center": ["W"] * 5,
    "valid_words": ["arrow;arrowroot;athwart;away;awry;harrow;thataway;thaw;throw;throwaway;thwart;wahoo;wart;warty;wary;watt;what;whoa;woot;worry;worrywart;wort;worth;worthy;wrath;yarrow"] * 5,
}

print("Format rewards: ", spelling_bee_format_reward(test_completions, **test_kwargs))
print("Letter rewards: ", spelling_bee_letter_reward(test_completions, **test_kwargs))
print("Dict rewards:   ", spelling_bee_dictionary_reward(test_completions, **test_kwargs))
print("Length rewards:  ", spelling_bee_length_reward(test_completions, **test_kwargs))
print("Pangram rewards:", spelling_bee_pangram_reward(test_completions, **test_kwargs))

Format rewards:  [1.0, 1.0, 0.0, 1.0, 1.0]
Letter rewards:  [1.0, 1.0, 0.0, 1.0, 1.0]
Dict rewards:    [3.0, 3.0, 0.0, 3.0, 3.0]
Length rewards:   [0.5, 0.5, 0.0, 0.1, 0.8]
Pangram rewards: [0.0, 0.0, 0.0, 0.0, 0.0]


## 4. Dataset Builder

### 4.1 Spelling Bee Dataset

In [ ]:
def load_spelling_bee_dataset(csv_path=None, max_puzzles=None):
    """Wrap the repo's SpellingBeeDataset into a HuggingFace Dataset with
    the extra columns the reward functions need (letters, center, valid_words)."""
    sb_ds = SpellingBeeDataset(csv_path=csv_path or _SB_CSV_PATH)
    rows = []
    for i, raw_row in enumerate(sb_ds._rows):
        if max_puzzles and i >= max_puzzles:
            break
        item = sb_ds[i]  # {"prompt": [...], "puzzle_id": int}
        rows.append({
            "prompt": item["prompt"],
            "letters": "".join(sorted(raw_row["letters"])),
            "center": raw_row["center"],
            "valid_words": ";".join(raw_row["solutions"]),
        })
    return Dataset.from_list(rows)


### 4.2 Wordle Multi-turn Dataset

bit different from other wordle loading in that has 4 variants per target (0,1,2,3 prior guesses) whereas previous `load_wordle_dataset` had 2 variants per target (0 or 1-4 prior guesses). also this one tracks green/gray/yellow letters to build more structured hints, whereas previous function formatted feedback string as tile emojis. 

In [9]:
def build_multiturn_wordle_dataset(max_words=50, max_guesses=6):
    """
    Pre-generate full game trajectories using decoy guesses.
    Each row is a full conversation up to turn N, 
    with the target as the final assistant message.
    """
    targets = [w.strip().upper() for w in 
               open(_WORDLE_PAST_SOLUTIONS_TXT_PATH).readlines() if w.strip()][:max_words]
    word_set = load_dictionary(length=5)
    decoy_pool = list(word_set - set(targets))
    rows = []
    
    for target in targets:
        # Generate 3 variants per target word
        for num_prior in [0, 1, 2, 3]:
            messages = [
                {"role": "system", "content": WORDLE_SYSTEM_PROMPT},
                {"role": "user", "content": WORDLE_USER_PROMPT_TEMPLATE.format(max_guesses=max_guesses)},
            ]
            
            green = {}
            gray = set()
            yellow = {}
            
            # Roll out num_prior random (non-solving) guesses
            prior_guesses = random.sample(decoy_pool, min(num_prior, len(decoy_pool)))
            for guess in prior_guesses:
                messages.append({"role": "assistant", "content": guess})
                
                tiles = _score_guess(guess, target)
                for pos, (letter, tile) in enumerate(zip(guess, tiles)):
                    if tile == "CORRECT":
                        green[pos] = letter
                    elif tile == "PRESENT":
                        yellow.setdefault(pos, set()).add(letter)
                    else:
                        gray.add(letter)
                
                # Add structured feedback
                hints = []
                if green:
                    hints.append("Confirmed: " + ", ".join(
                        f"{l} at position {p+1}" for p,l in sorted(green.items())))
                present = set(l for s in yellow.values() for l in s) - set(green.values())
                if present:
                    hints.append(f"In word but wrong position: {', '.join(sorted(present))}")
                truly_gray = gray - set(green.values()) - present
                if truly_gray:
                    hints.append(f"Not in word: {', '.join(sorted(truly_gray))}")
                
                fb = f"Feedback: {' | '.join(f'{l}:{t[:2]}' for l,t in zip(guess, tiles))}\n"
                if hints:
                    fb += "\n".join(hints)
                fb += f"\n{max_guesses - len(prior_guesses)} attempts remaining. Guess another 5-letter word."
                messages.append({"role": "user", "content": fb})
            
            rows.append({
                "prompt": messages,
                "game_type": "wordle",
                "target_word": target,
                # Pass constraint info for reward functions
                "known_greens": str(green),
                "known_grays": str(list(gray)),
            })
    
    return Dataset.from_list(rows)

### 4.3 Strands Multi-turn Dataset

In [10]:
def build_multiturn_strands_dataset(csv_path = None,max_puzzles=50):
    strands_ds = StrandsDataset(csv_path)
    rows = []
    
    for idx in range(min(max_puzzles, len(strands_ds))):
        raw = strands_ds._rows[idx]
        theme_words = raw["theme_words_list"]
        spangram = raw["spanagram"]
        
        # Variant 1: fresh start
        # Variant 2: 1 theme word already found
        # Variant 3: 2 theme words already found
        for num_found in [0, 1, 2]:
            messages = list(strands_ds[idx]["prompt"])
            already_found = theme_words[:num_found]
            
            for found_word in already_found:
                messages.append({"role": "assistant", "content": found_word})
                messages.append({"role": "user", "content": 
                    f"'{found_word}' is a theme word! +1 point.\n"
                    f"Found so far: {', '.join(already_found[:already_found.index(found_word)+1])}.\n"
                    f"Please guess another board word."})
            
            rows.append({
                "prompt": messages,
                "game_type": "strands",
                "theme_words": "|".join(theme_words),
                "spangram": spangram,
            })
    
    return Dataset.from_list(rows)

### 4.4 Connections Dataset

In [ ]:
def load_connections_dataset(csv_path=None, max_puzzles=None):
    conn_ds = ConnectionsDataset(csv_path=csv_path)
    rows = []
    for i in range(min(max_puzzles, len(conn_ds))):
        raw = conn_ds._rows[i]
        # Guard: skip any puzzle where a word is NaN (bad CSV row)
        categories = {
            cat: [str(w).strip().upper() for w in words if isinstance(w, str) or (hasattr(w, '__float__') and not __import__('math').isnan(float(w)))]
            for cat, words in raw["categories"].items()
        }
        # Skip incomplete puzzles after cleaning
        if any(len(v) != 4 for v in categories.values()):
            continue

        categories_str = "|".join(
            f"{cat}:{','.join(words)}"
            for cat, words in categories.items()
        )
        # Build prompt manually to avoid the NaN crash in __getitem__
        all_words = [w for words in categories.values() for w in words]
        import random as _random
        _random.shuffle(all_words)
        user_msg =CONNECTIONS_USER_PROMPT_TEMPLATE.format(
            words=", ".join(all_words)
        )
        rows.append({
            "prompt": [
                {"role": "system", "content": CONNECTIONS_SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            "game_type": "connections",
            "categories_str": categories_str,
        })
    return Dataset.from_list(rows)

### 4.5 Make Unified Dataset

In [ ]:
sb_dataset = load_spelling_bee_dataset(_SB_CSV_PATH, max_puzzles = MAX_PUZZLES)
wordle_dataset = build_multiturn_wordle_dataset()
strands_dataset = build_multiturn_strands_dataset(_STRANDS_CSV_PATH, max_puzzles = MAX_PUZZLES)
connections_dataset = load_connections_dataset(_CONNECTIONS_CSV_PATH, max_puzzles = MAX_PUZZLES)

# Add game_type to existing datasets that don't have it yet
sb_dataset = sb_dataset.map(lambda x: {"game_type": "spelling_bee"})
wordle_dataset = wordle_dataset.map(lambda x: {"game_type": "wordle"})


# Concatenate all 4 into one shuffled dataset
dataset = concatenate_datasets([
    sb_dataset,
    wordle_dataset,
    strands_dataset,
    connections_dataset,
]).shuffle(seed=42)

print(f"Total GRPO examples: {len(dataset)}")
from collections import Counter
print("Game distribution:", Counter(dataset["game_type"]))

Map: 100%|██████████| 100/100 [00:00<00:00, 45789.34 examples/s]

Total GRPO examples: 250
Game distribution: Counter({'wordle': 100, 'spelling_bee': 50, 'connections': 50, 'strands': 50})


## 5. Load Model & Tokenizer

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Dtype: {next(model.parameters()).dtype}")
print(f"Device: {next(model.parameters()).device}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Parameters: 1543.7M
Dtype: torch.bfloat16
Device: cuda:0


In [10]:
peft_config = None
if USE_LORA:
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    print(f"LoRA config: r={LORA_R}, alpha={LORA_ALPHA}")
    trainable = LORA_R * 2 * 7
    print(f"Estimated trainable params: ~{trainable * 28 / 1e3:.0f}K per attention block")
else:
    print("Full fine-tuning (no LoRA)")

LoRA config: r=16, alpha=32
Estimated trainable params: ~6K per attention block


## 5.5 SFT Warmstart

Before GRPO, do a quick supervised fine-tuning pass so the model learns what
valid Spelling Bee answers look like.

In [ ]:
from trl import SFTConfig, SFTTrainer

def build_sft_dataset(grpo_ds, max_per_game=75):
    counts = {"spelling_bee": 0, "wordle": 0, "strands": 0, "connections": 0}
    all_rows = []
    
    for item in grpo_ds:
        game_type = item["game_type"]
        if game_type not in counts:
            continue
        if counts[game_type] >= max_per_game:
            continue
        prompt = item["prompt"]

        if game_type == "spelling_bee":
            solutions = item["valid_words"].replace(";", ",").split(",")
            target = random.choice(solutions).strip()
            all_rows.append({"messages": prompt + [{"role": "assistant", "content": target}]})
            counts[game_type] += 1

        elif game_type == "wordle":
            all_rows.append({"messages": prompt + [{"role": "assistant", "content": item["target_word"]}]})
            counts[game_type] += 1

        elif game_type == "strands":
            all_rows.append({"messages": prompt + [{"role": "assistant", "content": item["spangram"]}]})
            counts[game_type] += 1

        elif game_type == "connections":
            for cat_entry in item["categories_str"].split("|"):
                if ":" not in cat_entry:
                    continue
                _, words_str = cat_entry.split(":", 1)
                all_rows.append({"messages": prompt + [{"role": "assistant", "content": words_str}]})
                counts[game_type] += 1
                break

    random.shuffle(all_rows)
    print(f"SFT examples per game: {counts}")
    return Dataset.from_list(all_rows)


sft_dataset = build_sft_dataset(grpo_ds = dataset, max_per_game = 75)
print(f"SFT dataset: {len(sft_dataset)} examples across all 4 games")
print(f"Sample: {sft_dataset[0]['messages'][-1]}")


SFT dataset: 1000 examples
Sample:
  Prompt: New Spelling Bee puzzle!
Allowed letters: A, B, C, E, K, L, M
Center letter (must appear in every wo...
  Answer: ABACK


In [ ]:
SFT_OUTPUT_DIR = "./sft_warmstart"
SFT_EPOCHS = 1    # reduced from 2 so that it would run shorter
SFT_BATCH_SIZE = 8
SFT_LR = 2e-5

sft_args = SFTConfig(
    output_dir=SFT_OUTPUT_DIR,
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    learning_rate=SFT_LR,
    bf16=USE_BF16,
    logging_steps=10,
    save_steps=100,
    save_total_limit=1,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(f"SFT warmstart: {len(sft_dataset)} examples, {SFT_EPOCHS} epoch(s)")
print(f"Estimated SFT steps: ~{len(sft_dataset) * SFT_EPOCHS // SFT_BATCH_SIZE}")

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

SFT warmstart: 1000 examples, 1 epoch(s)
Estimated SFT steps: ~250


In [13]:
# Run SFT warmstart — takes ~5-10 minutes on a T4
sft_trainer.train()
sft_trainer.save_model(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)
print(f"SFT warmstart model saved to {SFT_OUTPUT_DIR}")

# The model variable is now the SFT-warmed model (LoRA adapter applied in-place).
# GRPO training below will continue fine-tuning from this checkpoint.
# If using LoRA, merge SFT adapter so GRPO starts fresh with a new adapter.
if USE_LORA:
    model = sft_trainer.model.merge_and_unload()
    print("SFT LoRA merged into base model. GRPO will train a new LoRA adapter on top.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.170538
20,2.377165
30,1.794385
40,1.213871
50,0.636987
60,0.314119
70,0.195426
80,0.176992
90,0.168330
100,0.166074


SFT warmstart model saved to ./sft_warmstart
SFT LoRA merged into base model. GRPO will train a new LoRA adapter on top.


## 6. GRPO Training

In [14]:
training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,

    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    temperature=TEMPERATURE,
    beta=BETA,

    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    bf16=USE_BF16,
    logging_steps=1,
    save_steps=50,
    save_total_limit=3,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    report_to="none",
    log_level="info",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=reward_funcs,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print(f"Trainer created. Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Total training steps: ~{len(dataset) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM_STEPS)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Trainer created. Effective batch size: 8
Total training steps: ~75


In [15]:
trainer.train()

***** Running training *****
  Num examples = 200
  Num Epochs = 3
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 4
  Total optimization steps = 600
  Number of trainable parameters = 18,464,768
Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
1,0.539834
2,0.225590
3,0.359230
4,-0.018990
5,0.102675
6,0.036754
7,0.726026
8,0.098987
9,0.128193
10,0.103506


Saving model checkpoint to ./grpo_output/checkpoint-50
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",

TrainOutput(global_step=600, training_loss=0.11003860711236485, metrics={'train_runtime': 3676.6434, 'train_samples_per_second': 0.163, 'train_steps_per_second': 0.163, 'total_flos': 0.0, 'train_loss': 0.11003860711236485})

In [16]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Saving model checkpoint to ./grpo_output
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_att

Model saved to ./grpo_output


## 7. Inference & Testing with Spelling Bee Environment

In [17]:
print("SpellingBeeConfig:", SpellingBeeConfig)
print("SpellingBeeEnv:", SpellingBeeEnv)
print("WordleConfig:", WordleConfig)
print("WordleEnv:", WordleEnv)

SpellingBeeConfig: <class 'nytgames.env.spellingbee.SpellingBeeConfig'>
SpellingBeeEnv: <class 'nytgames.env.spellingbee.SpellingBeeEnv'>
WordleConfig: <class 'nytgames.env.wordle.WordleConfig'>
WordleEnv: <class 'nytgames.env.wordle.WordleEnv'>


In [ ]:
def load_trained_model(model_path):
    model_path = Path(model_path)
    adapter_config_path = model_path / "adapter_config.json"

    if adapter_config_path.exists():
        with open(adapter_config_path) as f:
            adapter_cfg = json.load(f)
        base_model_name = adapter_cfg.get("base_model_name_or_path", adapter_cfg.get("base_model"))
        print(f"Loading base model: {base_model_name}")
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
            # device_map="auto",
        )
        base_model = base_model.to("cuda" if torch.cuda.is_available() else "cpu")
        inf_model = PeftModel.from_pretrained(base_model, str(model_path))
        inf_tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    else:
        inf_model = AutoModelForCausalLM.from_pretrained(
            str(model_path),
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        inf_tokenizer = AutoTokenizer.from_pretrained(str(model_path))

    if inf_tokenizer.pad_token is None:
        inf_tokenizer.pad_token = inf_tokenizer.eos_token
    inf_model.eval()
    return inf_model, inf_tokenizer


def generate_guess(inf_model, inf_tokenizer, messages, temperature=0.7):
    text = inf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = inf_tokenizer(text, return_tensors="pt").to(inf_model.device)
    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens=32,
            temperature=temperature,
            do_sample=True,
            top_p=0.95,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return inf_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


inf_model, inf_tokenizer = load_trained_model(OUTPUT_DIR)

Loading base model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights: 100%|██████████| 338/338 [00:05<00:00, 64.54it/s, Materializing param=model.norm.weight]                              


### 7.1 Run a Spelling Bee Game

In [ ]:
config = SpellingBeeConfig(
    center_letter="H",
    letter_set={"G", "I", "P", "R", "A", "C", "H"},
    word_set={
        "GRAPHIC", "HIGHCHAIR", "PARAGRAPH", "ARCHAIC", "CHICHI", "PARIAH",
        "AARGH", "CHAIR", "CHICA", "CHIRP", "GRAPH", "PARCH",
        "ARCH", "CHAI", "CHAP", "CHAR", "CHIA", "CHIC", "CHIP",
        "HAIR", "HARP", "HIGH", "RICH",
    },
    max_guesses=20,
)

env = SpellingBeeEnv(config)
obs, info = env.reset()

prompt_text = SB_USER_PROMPT_TEMPLATE.format(
    letters=", ".join(sorted(config.letter_set)),
    center=config.center_letter,
    max_guesses=config.max_guesses,
)
messages = [
    {"role": "system", "content": SB_SYSTEM_PROMPT},
    {"role": "user", "content": prompt_text},
]

print(f"{'='*60}")
print(f"Spelling Bee — Letters: {sorted(config.letter_set)}, Center: {config.center_letter}")
print(f"Valid words: {len(config.word_set)}, Max guesses: {config.max_guesses}")
print(f"{'='*60}\n")

already_guessed = set()
allowed_letters = set(l.upper() for l in config.letter_set)
center = config.center_letter.upper()
MAX_RETRIES = 10  # more retries to filter out invalid-letter guesses
valid_pool = config.word_set
sorted_valid = sorted(valid_pool, key=len, reverse=True)
print(f"Valid pool: {len(valid_pool)} words, puzzle solutions: {len(config.word_set)}")

for turn in range(config.max_guesses):
    raw_guess = generate_guess(inf_model, inf_tokenizer, messages, temperature=1.0)
    candidate = "".join(c for c in raw_guess if c.isalpha()).upper()

    untried_valid = valid_pool - already_guessed

    if candidate in untried_valid:
        word = candidate
    elif untried_valid:
        # Pick next untried word in length order — not always the longest
        word = next((w for w in sorted_valid if w not in already_guessed), candidate)
    else:
        break  # exhausted all valid words, no point continuing

    already_guessed.add(word)
    messages.append({"role": "assistant", "content": word})
    obs, reward, terminated, truncated, info = env.step(word)

    print(f"Turn {turn+1:2d} | Guess: {word:15s} | Reward: {reward:2.0f} | Total: {obs['total_points']:3d}")

    if terminated or truncated:
        break

    if reward > 0:
        fb = f"'{word}' correct! +{reward:.0f} pts. Total: {obs['total_points']}."
    else:
        fb = f"'{word}' not accepted. {obs['feedback']}"

    guessed = ", ".join(obs["valid_words_guessed"]) or "none"
    fb += (f"\nWords found so far: {guessed}."
           f"\nOnly use: {', '.join(sorted(allowed_letters))} (center {center} required, min 4 letters)."
           f"\nGuess a different word you haven't tried yet.")
    messages.append({"role": "user", "content": fb})

print(f"\n{'='*60}")
print(f"Game Over! Final score: {obs['total_points']} in {obs['num_guesses']} guesses")
found_words = sorted(w for w, r, _ in info['history'] if r > 0)
print(f"Words found: {found_words}")
print(f"{'='*60}")

Spelling Bee — Letters: ['A', 'C', 'G', 'H', 'I', 'P', 'R'], Center: H
Valid words: 23, Max guesses: 20

Valid pool: 23 words, puzzle solutions: 23
Turn  1 | Guess: HIGHCHAIR       | Reward:  9 | Total:   9
Turn  2 | Guess: PARAGRAPH       | Reward:  9 | Total:  18
Turn  3 | Guess: ARCHAIC         | Reward:  7 | Total:  25
Turn  4 | Guess: GRAPHIC         | Reward: 14 | Total:  39
Turn  5 | Guess: CHICHI          | Reward:  6 | Total:  45
Turn  6 | Guess: PARIAH          | Reward:  6 | Total:  51
Turn  7 | Guess: CHIRP           | Reward:  5 | Total:  56
Turn  8 | Guess: GRAPH           | Reward:  5 | Total:  61
Turn  9 | Guess: AARGH           | Reward:  5 | Total:  66
Turn 10 | Guess: CHAIR           | Reward:  5 | Total:  71
Turn 11 | Guess: PARCH           | Reward:  5 | Total:  76
Turn 12 | Guess: CHICA           | Reward:  5 | Total:  81
Turn 13 | Guess: CHIA            | Reward:  1 | Total:  82
Turn 14 | Guess: CHAR            | Reward:  1 | Total:  83
Turn 15 | Guess: RICH     

### 7.2 Batch Evaluation

In [ ]:
def get_valid_sb_words(letter_set, center, dictionary, min_length=4):
    """Pre-compute all valid words for this puzzle — run once per puzzle."""
    allowed = set(l.upper() for l in letter_set)
    center = center.upper()
    return {w for w in dictionary 
            if len(w) >= min_length 
            and set(w) <= allowed 
            and center in w}

def evaluate_on_puzzles(inf_model, inf_tokenizer, csv_path=None, 
                        num_puzzles=10, max_guesses=10):
    sb_ds = SpellingBeeDataset(csv_path=csv_path or _SB_CSV_PATH, 
                               max_guesses=max_guesses)
    VALID_ENGLISH_WORDS = load_dictionary()  # load once outside loop
    results = []

    for idx in range(min(num_puzzles, len(sb_ds))):
        item   = sb_ds[idx]
        config = sb_ds.get_config(item["puzzle_id"], max_guesses=max_guesses)
        env    = SpellingBeeEnv(config)
        obs, _ = env.reset()
        messages = list(item["prompt"])

        allowed_letters = set(l.upper() for l in config.letter_set)
        center          = config.center_letter.upper()
        already_guessed = set()

        # Pre-compute valid words ONCE per puzzle instead of validating per attempt
        valid_pool = get_valid_sb_words(
            config.letter_set, config.center_letter, VALID_ENGLISH_WORDS
        )

        for _ in range(max_guesses):
            # Single model call — no retry loop
            raw       = generate_guess(inf_model, inf_tokenizer, messages, temperature=1.0)
            candidate = "".join(c for c in raw if c.isalpha()).upper()

            untried_valid = valid_pool - already_guessed

            if candidate in untried_valid:
                word = candidate
            elif untried_valid:
                # Fallback: pick longest untried valid word
                word = max(untried_valid, key=len)
            else:
                word = candidate  # exhausted all valid words

            already_guessed.add(word)
            messages.append({"role": "assistant", "content": word})

            obs, reward, terminated, truncated, info = env.step(word)

            fb = (f"'{word}' correct! Total: {obs['total_points']}."
                  if reward > 0 else
                  f"'{word}' not accepted. {obs['feedback']}")
            fb += (f"\nOnly use: {', '.join(sorted(allowed_letters))} "
                   f"(center {center} required)."
                   f"\nGuess another word, do not repeat previous guesses.")
            messages.append({"role": "user", "content": fb})

            if terminated or truncated:
                break

        letters_str = "".join(sorted(config.letter_set))
        results.append({
            "puzzle_id":       item["puzzle_id"],
            "letters":         letters_str,
            "center":          config.center_letter,
            "available_words": len(config.word_set),
            "found":           len(obs["valid_words_guessed"]),
            "score":           obs["total_points"],
        })
        print(f"Puzzle {item['puzzle_id']:3d} | {letters_str} (center={config.center_letter}) "
              f"| Found {len(obs['valid_words_guessed'])}/{len(config.word_set)} "
              f"| Score: {obs['total_points']}")

    avg_found = sum(r["found"] for r in results) / len(results)
    avg_score = sum(r["score"] for r in results) / len(results)
    print(f"\n{'='*60}")
    print(f"Evaluated {len(results)} puzzles")
    print(f"Avg words found: {avg_found:.1f}")
    print(f"Avg score: {avg_score:.1f}")
    print(f"{'='*60}")
    return results

eval_results = evaluate_on_puzzles(inf_model, inf_tokenizer, _SB_CSV_PATH, num_puzzles=100, max_guesses=10)

Puzzle   1 | AHORTWY (center=W) | Found 3/26 | Score: 34


KeyboardInterrupt: 

interrupted the above session, but typically gets around 2.6-3.0 words per puzzle with around a total score of 26 points per puzzle

## 8. Wordle Testing

In [ ]:
# Load the trained model for evaluation (shared across all game tests below)
if 'inf_model' not in dir():
    inf_model, inf_tokenizer = load_trained_model(OUTPUT_DIR)
else:
    print("inf_model already loaded, reusing.")


inf_model already loaded, reusing.


### 8.1 Run a Single Wordle Game

In [ ]:
# wrapping guesses in filter word function 
WORDLE_WORD_SET = load_dictionary(length=5)
target_word = "CRANE"

wordle_cfg = WordleConfig(target_word=target_word, word_set=WORDLE_WORD_SET, max_guesses=6)
env = WordleEnv(wordle_cfg)
obs, info = env.reset()

user_msg = WORDLE_USER_PROMPT_TEMPLATE.format(max_guesses=wordle_cfg.max_guesses)
messages = [
    {"role": "system", "content": WORDLE_SYSTEM_PROMPT},
    {"role": "user", "content": user_msg},
]

print(f"{'='*60}")
print(f"Wordle — Target: {target_word} (hidden from model)")
print(f"{'='*60}\n")

known_green = {}
known_gray = set()

def filtered_wordle_guess(messages, known_green, known_gray):
    raw = generate_guess(inf_model, inf_tokenizer, messages, temperature=0.7)
    guess = "".join(c for c in raw if c.isalpha()).upper()[:5]
    
    valid = [w for w in WORDLE_WORD_SET
             if len(w) == 5
             and all(w[p] == l for p, l in known_green.items())
             and not any(l in w for l in known_gray)]
    
    if guess in valid:
        return guess
    elif valid:
        return random.choice(valid[:20])
    return guess

for turn in range(wordle_cfg.max_guesses):
    word = filtered_wordle_guess(messages, known_green, known_gray)
    messages.append({"role": "assistant", "content": word})
    obs, reward, terminated, truncated, info = env.step(word)

    fb_line = obs["feedback"] if obs["feedback"] else "(invalid)"
    print(f"Turn {turn+1} | Guess: {word:5s} | {fb_line} | Reward: {reward:.0f}")

    if terminated:
        print(f"\n✅ Solved in {obs['num_guesses']} guesses!")
        break
    if truncated:
        print(f"\n❌ Failed to guess '{target_word}' in {wordle_cfg.max_guesses} tries.")
        break

    # Update constraints
    if obs["feedback"] and "Invalid" not in obs["feedback"]:
        tiles = _score_guess(word, target_word)
        for pos, (letter, tile) in enumerate(zip(word, tiles)):
            if tile == "CORRECT":
                known_green[pos] = letter
            elif tile == "ABSENT":
                known_gray.add(letter)

    # Build feedback message
    hints = []
    if known_green:
        hints.append("Confirmed: " + ", ".join(f"{l} at position {p+1}" 
                     for p, l in sorted(known_green.items())))
    if known_gray:
        hints.append(f"Not in word: {', '.join(sorted(known_gray))}")
    
    fb_msg = f"Feedback: {obs['feedback']}\nYou have {wordle_cfg.max_guesses - obs['num_guesses']} attempts remaining."
    if hints:
        fb_msg += "\n\nWhat we know:\n" + "\n".join(hints)
    fb_msg += "\nPlease guess another 5-letter word."
    messages.append({"role": "user", "content": fb_msg})

env.render()

Wordle — Target: CRANE (hidden from model)

Turn 1 | Guess: QUIRK | ⬜⬜⬜🟨⬜  QUIRK | Reward: 1
Turn 2 | Guess: LIMES | ⬜⬜⬜🟨⬜  LIMES | Reward: 1
Turn 3 | Guess: SILEN | ⬜⬜⬜🟨🟨  SILEN | Reward: 1
Turn 4 | Guess: BURNS | ⬜⬜🟨🟩⬜  BURNS | Reward: 1
Turn 5 | Guess: CRIES | 🟩🟩⬜🟨⬜  CRIES | Reward: 1
Turn 6 | Guess: CREAM | 🟩🟩🟨🟨⬜  CREAM | Reward: 1
Turn 7 | Guess: CRANE | 🟩🟩🟩🟩🟩  CRANE - Correct! | Reward: 16

✅ Solved in 7 guesses!

Wordle | Guess 7/10
⬜⬜⬜🟨⬜  QUIRK
⬜⬜⬜🟨⬜  LIMES
⬜⬜⬜🟨🟨  SILEN
⬜⬜🟨🟩⬜  BURNS
🟩🟩⬜🟨⬜  CRIES
🟩🟩🟨🟨⬜  CREAM
🟩🟩🟩🟩🟩  CRANE
⬜⬜⬜⬜⬜
⬜⬜⬜⬜⬜
⬜⬜⬜⬜⬜

Total points: 22 
Keyboard: A🟩 B⬜ C🟩 D  E🟩 F  G  H  I⬜ J  K⬜ L⬜ M⬜ N🟩 O  P  Q⬜ R🟩 S⬜ T  U⬜ V  W  X  Y  Z 


supposed to limit to 6 guesses, but unsuccessful in guessing target word within that limit. above is example of a successful guess when given more guesses. 

### 8.2 Batch Wordle Evaluation

In [ ]:
def evaluate_wordle(inf_model, inf_tokenizer, num_puzzles=20, max_guesses=6,
                    solutions_path=None):
    """Play full Wordle games against past NYT solutions and report stats."""
    solutions_path = solutions_path or _WORDLE_PAST_SOLUTIONS_TXT_PATH
    words = [w.strip().upper() for w in open(solutions_path).readlines() if w.strip()]
    words = words[:num_puzzles]

    word_set = load_dictionary(length=5)
    results = []

    for target in words:
        cfg = WordleConfig(target_word=target, word_set=word_set, max_guesses=max_guesses)
        env = WordleEnv(cfg)
        obs, info = env.reset()

        user_msg = WORDLE_USER_PROMPT_TEMPLATE.format(max_guesses=max_guesses)
        messages = [
            {"role": "system", "content": WORDLE_SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ]

        green = {}
        yellow = {}
        gray = set()

        for _ in range(max_guesses):
            word = filtered_wordle_guess(messages, green, gray)

            messages.append({"role": "assistant", "content": word})
            obs, reward, terminated, truncated, info = env.step(word)

            if terminated or truncated:
                break

            # Update constraints
            if obs["feedback"] and "Invalid" not in obs["feedback"]:
                tiles = _score_guess(word, target)
                for pos, (letter, tile) in enumerate(zip(word, tiles)):
                    if tile == "CORRECT":
                        green[pos] = letter
                    elif tile == "PRESENT":
                        yellow.setdefault(pos, set()).add(letter)
                    else:
                        gray.add(letter)

            # Build constraint summary
            hints = []
            if green:
                hints.append("Confirmed: " + ", ".join(f"{l} at position {p+1}" for p, l in sorted(green.items())))
            present = set()
            for s in yellow.values():
                present.update(s)
            present -= set(green.values())
            if present:
                hints.append(f"In word but position unknown: {', '.join(sorted(present))}")
            truly_gray = gray - set(green.values()) - present
            if truly_gray:
                hints.append(f"Not in word: {', '.join(sorted(truly_gray))}")

            fb_msg = f"Feedback: {obs['feedback']}\nYou have {max_guesses - obs['num_guesses']} attempts remaining."
            if hints:
                fb_msg += "\n\nWhat we know:\n" + "\n".join(hints)
            fb_msg += "\nPlease guess another 5-letter word."
            messages.append({"role": "user", "content": fb_msg})

        solved = obs["solved"]
        guesses_used = obs["num_guesses"]
        results.append({
            "target": target,
            "solved": solved,
            "guesses_used": guesses_used,
            "score": obs["total_points"],
        })
        status = f"✅ {guesses_used}/6" if solved else "❌ FAIL"
        print(f"{target}  {status}  | Score: {obs['total_points']}")

    num_solved = sum(r["solved"] for r in results)
    avg_guesses = (sum(r["guesses_used"] for r in results if r["solved"]) / max(num_solved, 1))
    avg_score = sum(r["score"] for r in results) / len(results)

    print(f"\n{'='*60}")
    print(f"Evaluated {len(results)} Wordle puzzles")
    print(f"Solved: {num_solved}/{len(results)} ({100*num_solved/len(results):.1f}%)")
    print(f"Avg guesses (solved only): {avg_guesses:.2f}")
    print(f"Avg score: {avg_score:.1f}")
    print(f"{'='*60}")
    return results


wordle_results = evaluate_wordle(inf_model, inf_tokenizer, num_puzzles=100, max_guesses=6)

ABACK  ❌ FAIL  | Score: 7
ABASE  ❌ FAIL  | Score: 7
ABATE  ❌ FAIL  | Score: 9
ABBEY  ❌ FAIL  | Score: 9
ABBOT  ❌ FAIL  | Score: 8
ABHOR  ❌ FAIL  | Score: 7
ABIDE  ❌ FAIL  | Score: 9
ABOUT  ❌ FAIL  | Score: 9
ABOVE  ❌ FAIL  | Score: 7
ABYSS  ❌ FAIL  | Score: 8
ACORN  ❌ FAIL  | Score: 9
ACRID  ❌ FAIL  | Score: 6
ACTOR  ❌ FAIL  | Score: 8
ACUTE  ❌ FAIL  | Score: 10
ADAGE  ❌ FAIL  | Score: 9
ADAPT  ❌ FAIL  | Score: 7
ADEPT  ❌ FAIL  | Score: 8
ADMIN  ❌ FAIL  | Score: 9
ADMIT  ❌ FAIL  | Score: 8
ADOBE  ❌ FAIL  | Score: 8

Evaluated 20 Wordle puzzles
Solved: 0/20 (0.0%)
Avg guesses (solved only): 0.00
Avg score: 8.1


## 9. Strands Testing

### 9.1 Run a Single Strands Game

In [57]:
# Pick a specific puzzle to test (change index to try others)
TEST_PUZZLE_IDX = 0

strands_ds_eval = StrandsDataset()
strands_item    = strands_ds_eval[TEST_PUZZLE_IDX]
strands_raw     = strands_ds_eval._rows[TEST_PUZZLE_IDX]

strands_cfg = StrandsConfig(
    theme        = strands_raw["clue"],
    theme_words  = set(strands_raw["theme_words_list"]),
    spanagram    = strands_raw["spanagram"],
    board        = strands_raw["board"],
    dictionary   = load_dictionary(),
)
strands_env = StrandsEnv(strands_cfg)
obs, info   = strands_env.reset()
messages    = list(strands_item["prompt"])

all_theme_words = set(w.upper() for w in strands_raw["theme_words_list"])
spangram        = strands_raw["spanagram"].upper()
already_guessed = set()
max_turns       = len(all_theme_words) * 3  # generous budget

print(f"{'='*60}")
print(f"Strands — Theme: {strands_cfg.theme}")
print(f"Theme words ({len(all_theme_words)}): {sorted(all_theme_words)}")
print(f"Spangram: {spangram}")
print(f"{'='*60}\n")

found_words = []
for turn in range(max_turns):
    word = None
    for attempt in range(10):
        raw = generate_guess(inf_model, inf_tokenizer, messages, temperature=1.0)
        candidate = raw.split()[0] if raw.split() else raw
        candidate = "".join(c for c in candidate if c.isalpha()).upper()
        if candidate and candidate not in already_guessed:
            word = candidate
            break
    if word is None:
        word = candidate or "SKIP"
    already_guessed.add(word)

    messages.append({"role": "assistant", "content": word})
    obs, reward, terminated, truncated, info = strands_env.step(word)

    if word == spangram:
        tag = "🌟 SPANGRAM"
        found_words.append(word)
    elif word in all_theme_words:
        tag = "✅ THEME WORD"
        found_words.append(word)
    else:
        tag = "❌"
    print(f"Turn {turn+1:2d} | {word:15s} | {tag}")

    if terminated or truncated:
        break

    fb = obs.get("feedback", "Not a theme word.")
    messages.append({"role": "user", "content":
        f"'{word}': {fb}\nFound so far: {', '.join(found_words) or 'none'}.\nPlease guess another board word."})

print(f"\n{'='*60}")
print(f"Found {len(found_words)}/{len(all_theme_words)} theme words: {found_words}")
print(f"Spangram found: {spangram in found_words}")
print(f"{'='*60}")


Strands — Theme: Mark my words
Theme words (6): ['APOSTROPHE', 'COLON', 'COMMA', 'HYPHEN', 'PERIOD', 'SLASH']
Spangram: PUNCTUATION



KeyboardInterrupt: 

interrupted this session, but in previous runs it fails to solve this single puzzle or guess any of the words within it

### 9.2 Batch Strands Evaluation

In [ ]:
TEST_PUZZLE_IDX = 0

strands_ds_eval = StrandsDataset()
strands_item    = strands_ds_eval[TEST_PUZZLE_IDX]
strands_raw     = strands_ds_eval._rows[TEST_PUZZLE_IDX]

strands_cfg = StrandsConfig(
    theme        = strands_raw["clue"],
    theme_words  = set(strands_raw["theme_words_list"]),
    spanagram    = strands_raw["spanagram"],
    board        = strands_raw["board"],
    dictionary   = load_dictionary(),
)
strands_env = StrandsEnv(strands_cfg)
obs, info   = strands_env.reset()
messages    = list(strands_item["prompt"])

all_theme_words = set(w.upper() for w in strands_raw["theme_words_list"])
spangram        = strands_raw["spanagram"].upper()
already_guessed = set()
max_turns       = len(all_theme_words) * 3  # generous budget

VALID_ENGLISH_WORDS = load_dictionary()
WORD_PREFIXES = set()
for word in VALID_ENGLISH_WORDS:
    for i in range(1, len(word)+1):
        WORD_PREFIXES.add(word[:i])

def get_valid_board_words(board, min_length=4, max_length=12):
    rows, cols = len(board), len(board[0])
    found = set()
    
    def dfs(r, c, visited, word):
        if word not in WORD_PREFIXES:
            return
        if len(word) >= min_length and word in VALID_ENGLISH_WORDS:
            found.add(word)
        if len(word) >= max_length:
            return
        for dr, dc in [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr,nc) not in visited:
                dfs(nr, nc, visited|{(nr,nc)}, word + board[nr][nc])
    
    for r in range(rows):
        for c in range(cols):
            dfs(r, c, {(r,c)}, board[r][c])
    return found

# filter guesses to valid board words
VALID_ENGLISH_WORDS = load_dictionary()
valid_board_words = get_valid_board_words(strands_raw["board"])

# Override generate_guess to only return valid board words
def filtered_strands_guess(messages, valid_board_words, already_guessed,theme_hint, inf_model, inf_tokenizer):
    raw = generate_guess(inf_model, inf_tokenizer, messages, temperature=1.0)
    candidate = "".join(c for c in raw if c.isalpha()).upper()
    
    untried = valid_board_words - already_guessed
    if not untried:
        return candidate
    
    if candidate in untried:
        return candidate
    
    # Theme words are almost always longer than noise words
    return max(untried, key=len)

def evaluate_strands(inf_model, inf_tokenizer, num_puzzles=10):
    strands_ds = StrandsDataset()
    results = []

    for idx in range(min(num_puzzles, len(strands_ds))):
        item = strands_ds[idx]
        raw  = strands_ds._rows[idx]
        cfg  = StrandsConfig(
            theme       = raw["clue"],
            theme_words = set(raw["theme_words_list"]),
            spanagram   = raw["spanagram"],
            board       = raw["board"],
            dictionary  = VALID_ENGLISH_WORDS,  # reuse, don't reload
        )
        env      = StrandsEnv(cfg)
        obs, _   = env.reset()
        messages = list(item["prompt"])

        all_words       = set(w.upper() for w in raw["theme_words_list"])
        spangram        = raw["spanagram"].upper()
        already_guessed = set()
        found_words     = []
        max_turns       = len(all_words) * 3

        # Uses global prefixes — fast
        valid_board_words = get_valid_board_words(raw["board"])

        for _ in range(max_turns):
            word = filtered_strands_guess(
                messages, valid_board_words, already_guessed,
                cfg.theme, inf_model, inf_tokenizer
            )
            already_guessed.add(word)
            messages.append({"role": "assistant", "content": word})
            obs, reward, terminated, truncated, _ = env.step(word)

            if word in all_words:
                found_words.append(word)
            if terminated or truncated:
                break

            fb = obs.get("feedback", "Not a theme word.")
            messages.append({"role": "user", "content":
                f"'{word}': {fb}\nFound: {', '.join(found_words) or 'none'}.\n"
                f"Guess another board word."})

        spangram_found = spangram in found_words
        results.append({
            "theme": raw["clue"], "found": len(found_words),
            "total": len(all_words), "spangram": spangram_found,
        })
        span_icon = "🌟" if spangram_found else "  "
        print(f"{span_icon} {raw['clue']:30s} | Found: {len(found_words)}/{len(all_words)}")

    avg_found = sum(r["found"] for r in results) / len(results)
    spangrams = sum(r["spangram"] for r in results)
    print(f"\n{'='*60}")
    print(f"Evaluated {len(results)} Strands puzzles")
    print(f"Avg theme words found: {avg_found:.1f}")
    print(f"Spangrams found: {spangrams}/{len(results)}")
    print(f"{'='*60}")
    return results

strands_results = evaluate_strands(inf_model, inf_tokenizer, num_puzzles=100)



   Mark my words                  | Found: 1/6
   She’ll have a ball             | Found: 0/6
   I gotta dip!                   | Found: 0/6
   Don’t do it!                   | Found: 0/7
   Do the do                      | Found: 1/6
   FRAGILE: handle with care      | Found: 0/7
   Sounds good to me              | Found: 0/6
   To put it mildly               | Found: 1/7
   Ruler’s decree                 | Found: 0/7
   One thousand followers         | Found: 0/7
   Get animated                   | Found: 1/6
   Old haunts                     | Found: 0/6
   Grow up!                       | Found: 0/8
   Outside interests              | Found: 0/7
   World piece                    | Found: 0/7
   That’s fortunate               | Found: 3/5
   Back and forth                 | Found: 1/7
   What’s the issue?              | Found: 1/6
   Animal sounds                  | Found: 0/7
   Romeo and Juliet               | Found: 0/7
   Lock steps                     | Found: 1/6
   Sign langu

## 10. Connections Testing

### 10.1 Run a Single Connections Game

In [ ]:
# testing what would happen if change system prompt to this instead
CONNECTIONS_SYSTEM = """You are playing NYT Connections.
Respond with EXACTLY 4 words separated by commas. Nothing else.
Example: APPLE, BANANA, CHERRY, DATE
No reasoning. No tags. Just 4 words."""

In [48]:
TEST_PUZZLE_IDX = 0

conn_ds_eval = ConnectionsDataset()
conn_item    = conn_ds_eval[TEST_PUZZLE_IDX]
conn_raw     = conn_ds_eval._rows[TEST_PUZZLE_IDX]

categories = {cat: frozenset(w.upper() for w in words)
              for cat, words in conn_raw["categories"].items()}
all_words  = set(w for words in categories.values() for w in words)

conn_cfg = ConnectionsConfig(categories=conn_raw["categories"], max_mistakes=4)
conn_env = ConnectionsEnv(conn_cfg)
obs, _   = conn_env.reset()

words_display = ", ".join(sorted(all_words))
messages = [
    {"role": "system", "content": CONNECTIONS_SYSTEM},
    {"role": "user", "content":
        f"Here are 16 words: {words_display}\n"
        f"Guess 4 words that belong to the same category.\n"
        f"Reply with exactly 4 comma-separated words, nothing else."}
]

remaining_cats = dict(categories)
solved_cats    = []
mistakes       = 0
max_turns      = conn_cfg.max_mistakes + len(categories)

print(f"{'='*60}")
print(f"Connections Puzzle {conn_raw['puzzle_id']}")
for cat, words in categories.items():
    print(f"  [{cat}]: {sorted(words)}")
print(f"{'='*60}\n")

def connections_guess_with_hint(inf_model, inf_tokenizer, remaining_words, temperature=0.3):
    """Ask model to group from the actual remaining words only."""
    words_list = sorted(remaining_words)
    prompt = (
        f"These {len(words_list)} words need to be split into groups of 4 "
        f"that share a hidden connection:\n"
        f"{', '.join(words_list)}\n\n"
        f"Pick 4 words that most likely belong together.\n"
        f"Reply with ONLY 4 words separated by commas. Example: WORD1, WORD2, WORD3, WORD4"
    )
    messages = [
        {"role": "system", "content": "You are an expert at NYT Connections. "
                                       "Only use words from the provided list. "
                                       "Reply with exactly 4 comma-separated words."},
        {"role": "user", "content": prompt}
    ]
    raw = generate_guess(inf_model, inf_tokenizer, messages, temperature=temperature)
    
    # Parse and validate against remaining words
    candidates = [w.strip().upper() for w in raw.split(",")]
    valid = [w for w in candidates if w in remaining_words]
    
    if len(valid) == 4:
        return valid
    
    # If still invalid, try with lower temperature
    raw2 = generate_guess(inf_model, inf_tokenizer, messages, temperature=0.1)
    candidates2 = [w.strip().upper() for w in raw2.split(",")]
    valid2 = [w for w in candidates2 if w in remaining_words]
    
    if len(valid2) == 4:
        return valid2
    
    # Final fallback — return None to signal we should skip
    return None


for turn in range(max_turns):
    if mistakes >= conn_cfg.max_mistakes:
        print("Max mistakes reached — stopping.")
        break

    remaining_words = set(w for words in remaining_cats.values() for w in words)
    
    # Always use the constrained guesser — never rely on raw model output
    valid_guessed = connections_guess_with_hint(
        inf_model, inf_tokenizer, remaining_words, temperature=0.3
    )
    
    if valid_guessed is None:
        # Absolute fallback — pick most common letter pattern group
        sorted_remaining = sorted(remaining_words)
        valid_guessed = sorted_remaining[:4]
        print(f"  (fallback to first 4 alphabetically)")

    guess_str = ", ".join(valid_guessed)
    messages.append({"role": "assistant", "content": guess_str})
    obs, reward, terminated, truncated, _ = conn_env.step(guess_str)

    guessed_set = frozenset(valid_guessed)
    matched_cat = next((cat for cat, words in remaining_cats.items()
                        if guessed_set == words), None)

    if matched_cat:
        solved_cats.append(matched_cat)
        del remaining_cats[matched_cat]
        print(f"Turn {turn+1:2d} | ✅ '{matched_cat}': {sorted(guessed_set)}")
    else:
        mistakes += 1
        print(f"Turn {turn+1:2d} | ❌ Wrong: {guess_str}  (mistake {mistakes}/{conn_cfg.max_mistakes})")

    if terminated or truncated or mistakes >= conn_cfg.max_mistakes:
        break

    fb_remaining = sorted(w for words in remaining_cats.values() for w in words)
    messages.append({"role": "user", "content":
        f"{'Correct!' if matched_cat else 'Wrong, try again.'}\n"
        f"Solved: {solved_cats or 'none'}.\n"
        f"Remaining: {', '.join(fb_remaining)}.\n"
        f"Pick 4 more words that belong together."})

print(f"\n{'='*60}")
print(f"Solved {len(solved_cats)}/{len(categories)} categories: {solved_cats}")
print(f"Mistakes: {mistakes}/{conn_cfg.max_mistakes}")
print(f"{'='*60}")


Connections Puzzle 1
  [KEYBOARD KEYS]: ['OPTION', 'RETURN', 'SHIFT', 'TAB']
  [NBA TEAMS]: ['BUCKS', 'HEAT', 'JAZZ', 'NETS']
  [PALINDROMES]: ['KAYAK', 'LEVEL', 'MOM', 'RACECAR']
  [WET WEATHER]: ['HAIL', 'RAIN', 'SLEET', 'SNOW']

  (fallback to first 4 alphabetically)
Turn  1 | ❌ Wrong: BUCKS, HAIL, HEAT, JAZZ  (mistake 1/4)
Turn  2 | ❌ Wrong: HEAT, RAIN, SNOW, TAB  (mistake 2/4)
Turn  3 | ❌ Wrong: HEAT, JAZZ, OPTION, SNOW  (mistake 3/4)
Turn  4 | ❌ Wrong: HAIL, HEAT, RAIN, SNOW  (mistake 4/4)

Solved 0/4 categories: []
Mistakes: 4/4


### 10.2 Batch Connections Evaluation

In [59]:
def evaluate_connections(inf_model, inf_tokenizer, num_puzzles=10):
    conn_ds = ConnectionsDataset()
    results = []

    for idx in range(min(num_puzzles, len(conn_ds))):
        item     = conn_ds[idx]
        raw      = conn_ds._rows[idx]
        cfg      = ConnectionsConfig(categories=raw["categories"], max_mistakes=4)
        env      = ConnectionsEnv(cfg)
        obs, _   = env.reset()
        messages = list(item["prompt"])

        categories     = {cat: frozenset(w.upper() for w in words)
                          for cat, words in raw["categories"].items()}
        remaining_cats = dict(categories)
        solved_cats    = []
        mistakes       = 0
        max_turns      = cfg.max_mistakes + len(categories)

        for _ in range(max_turns):
            guess_str = generate_guess(inf_model, inf_tokenizer, messages, temperature=0.7).strip()
            messages.append({"role": "assistant", "content": guess_str})
            obs, reward, terminated, truncated, _ = env.step(guess_str)

            guessed_words = frozenset(w.strip().upper() for w in guess_str.split(","))
            matched_cat   = next((cat for cat, words in remaining_cats.items()
                                  if guessed_words == words), None)
            if matched_cat:
                solved_cats.append(matched_cat)
                del remaining_cats[matched_cat]
            else:
                mistakes += 1

            if terminated or truncated:
                break

            remaining_words = sorted(w for words in remaining_cats.values() for w in words)
            fb = obs.get("feedback", "That group is not correct.")
            messages.append({"role": "user", "content":
                f"{fb}\nSolved: {solved_cats or 'none'}.\n"
                f"Remaining words: {', '.join(remaining_words)}.\n"
                "Guess 4 words for the same category, comma-separated."})

        perfect = len(solved_cats) == len(categories)
        results.append({"puzzle_id": raw.get("puzzle_id", idx),
                         "solved": len(solved_cats), "total": len(categories),
                         "mistakes": mistakes, "perfect": perfect})
        icon = "✅" if perfect else f"{len(solved_cats)}/{len(categories)}"
        print(f"Puzzle {raw.get('puzzle_id', idx):>4} | {icon} | Mistakes: {mistakes}")

    perfect_count = sum(r["perfect"] for r in results)
    avg_solved    = sum(r["solved"]  for r in results) / len(results)
    print(f"\n{'='*60}")
    print(f"Evaluated {len(results)} Connections puzzles")
    print(f"Perfect solves: {perfect_count}/{len(results)}")
    print(f"Avg categories solved: {avg_solved:.1f} / {results[0]['total']}")
    print(f"{'='*60}")
    return results


connections_results = evaluate_connections(inf_model, inf_tokenizer, num_puzzles=100)


Puzzle    1 | 0/4 | Mistakes: 8
Puzzle    2 | 0/4 | Mistakes: 8
Puzzle    3 | 0/4 | Mistakes: 8
Puzzle    4 | 0/4 | Mistakes: 8
Puzzle    5 | 0/4 | Mistakes: 8
Puzzle    6 | 0/4 | Mistakes: 8
Puzzle    7 | 0/4 | Mistakes: 8
Puzzle    8 | 0/4 | Mistakes: 8
Puzzle    9 | 0/4 | Mistakes: 8
Puzzle   10 | 0/4 | Mistakes: 8
Puzzle   11 | 0/4 | Mistakes: 8
Puzzle   12 | 0/4 | Mistakes: 8
Puzzle   13 | 0/4 | Mistakes: 8
Puzzle   14 | 0/4 | Mistakes: 8
Puzzle   15 | 0/4 | Mistakes: 8
Puzzle   16 | 0/4 | Mistakes: 8
Puzzle   17 | 0/4 | Mistakes: 8
Puzzle   18 | 0/4 | Mistakes: 8
Puzzle   19 | 0/4 | Mistakes: 8
Puzzle   20 | 0/4 | Mistakes: 8
Puzzle   21 | 0/4 | Mistakes: 8
Puzzle   22 | 0/4 | Mistakes: 8
Puzzle   23 | 0/4 | Mistakes: 8
Puzzle   24 | 0/4 | Mistakes: 8
Puzzle   25 | 0/4 | Mistakes: 8
Puzzle   26 | 0/4 | Mistakes: 8
Puzzle   27 | 0/4 | Mistakes: 8
Puzzle   28 | 0/4 | Mistakes: 8
Puzzle   29 | 0/4 | Mistakes: 8
Puzzle   30 | 0/4 | Mistakes: 8
Puzzle   31 | 0/4 | Mistakes: 8
Puzzle  

TypeError: sequence item 10: expected str instance, float found

fails to solve any of the 4 categories within any tested connections puzzle